# Part 2 — Pointcept + PTv3 baselines (local GPU, VS Code)
Point-cloud segmentation + super resolution · S3DIS · PTv3 (RGB only)

**Prerequisites:** the conda environment from the README (Python 3.10, PyTorch 2.5.0 + CUDA 12.4, spconv) is the kernel of this notebook, and the Part 1 data is in `workspace/data/` (`s3dis`, `s3dis_sparse25`, optionally `s3dis_sparse12`).
Everything is resumable: if the notebook, VS Code or the computer stops, run the same training cell again and it continues from the last checkpoint.

What this notebook does
1. checks the environment and the data, clones Pointcept (pinned commit) and applies the small compatibility patch,
2. builds the experiment configs from one shared settings file, measures training speed, and runs **Phase 0** (does sparsity hurt?) and the baselines **A, B, C, D**.

Design decisions you should state in your report
- PTv3 uses **RGB only** (no normals): the data has no normals and upsampled points would have none either. Absolute mIoU is therefore lower than the paper's.
- Every variant uses the **same schedule** (the knobs in section 3); compare variants with each other, not with the paper.
- `evaluate=False`: **no validation on Area 5 during training**, the **last checkpoint** is used (no model selection on the test set).
- Every variant is **scored on the original dense Area 5 points** (predictions are mapped back by 1-NN), so numbers are directly comparable.

## 0. Setup and data check

In [ ]:
import sys
from pathlib import Path
PROJECT = Path.cwd().resolve()
if not (PROJECT / "pcseg").is_dir():          # notebook opened from the notebooks/ folder
    PROJECT = PROJECT.parent
sys.path.insert(0, str(PROJECT))

import json, os, shutil
import numpy as np
import torch
from pcseg import settings, run, env_setup, results
settings.ensure_dirs()
D, S25, S12 = str(settings.DENSE), str(settings.SP25), str(settings.SP12)

g = settings.gpu_info()
print("torch", torch.__version__, "| cuda", torch.version.cuda, "| gpu:", f"{g[0]} ({g[1]:.0f} GB)" if g else "NONE")
print("workspace:", settings.ROOT)
print("data     :", settings.DATA)
print(f"free disk: {shutil.disk_usage(settings.ROOT).free / 2**30:.0f} GB")

In [ ]:
assert torch.cuda.is_available(), "No GPU visible to PyTorch: see README, Troubleshooting"
from pcseg import data_check
assert data_check.check(), "fix the data problems listed above (README, step 3) before continuing"

## 1. Environment check
Verifies Python / PyTorch / CUDA / spconv on your GPU and whether fp16 (mixed precision, AMP) works. The result is stored in `workspace/pcseg_settings.json` and used by every later run.

In [ ]:
env_setup.check_env()
K = settings.load_knobs()
K.amp = env_setup.check_amp()        # True = mixed precision, False = fp32 (slower, more memory)
K.save()
print("AMP:", K.amp)

## 2. Pointcept + compatibility layer

In [ ]:
env_setup.prepare_pointcept()      # clone Pointcept (pinned commit), add RandomSparsify, shim/stub what cannot be imported

In [ ]:
env_setup.import_test()             # the Pointcept modules that train/test import must load without errors

## 3. Training budget (knobs) and configs
These knobs are written to `workspace/pcseg_settings.json` and read by **every** run in Parts 2 and 4, so rows A, B, D and H always use the same budget. Do not change them after the first full run.
- `batch`, `point_max`: lower them if you hit CUDA out-of-memory. The suggestion below is an estimate from your VRAM, scaled from the 16 GB T4 setting; the benchmark (section 4) is what decides.
- `epoch`: total passes over the training rooms (paper: 3000). `eval_epoch`: number of checkpoints written (one every `epoch // eval_epoch` passes). `epoch` must be a multiple of it.
- Learning rate is scaled linearly with the batch size (paper: 0.006 at batch 12).
- If you copied finished Colab results into the workspace, set the knobs to the Colab values (4, 100000, 100, 10) so that new rows stay comparable with them.

In [ ]:
from pcseg import experiments
from pcseg.make_configs import iterations

K = settings.load_knobs()
if g:
    print("suggested for your GPU:", settings.suggest_knobs(g[1]))

# ------------------------------------------------------------- edit the knobs here (then run the cell)
K.batch, K.point_max, K.epoch, K.eval_epoch = 4, 100000, 100, 10
K.num_worker = 4                      # data loader workers; raise it if the GPU waits for data (see GPU utilisation)
# -------------------------------------------------------------------------------------------------
K.save()

TRAIN_AREAS = ['Area_1', 'Area_2', 'Area_3', 'Area_4', 'Area_6']
N_TRAIN_ROOMS = sum(len(os.listdir(f'{D}/{a}')) for a in TRAIN_AREAS)
print(N_TRAIN_ROOMS, 'training rooms,', len(os.listdir(f'{D}/Area_5')), 'Area_5 rooms')
cfgs = experiments.make_all(K)
print({k: Path(v).name for k, v in cfgs.items()})
print('iterations per full run:', iterations(N_TRAIN_ROOMS, K.batch, K.epoch), '| AMP:', K.amp, '| knobs:', K)

In [ ]:
env_setup.smoke_test(cfgs["smoke"], K.amp)       # one real PTv3 forward pass on the GPU (expects: logits (20000, 13))

## 4. Benchmark: how fast is one iteration?
Runs one short pass (about 50 iterations). If it fails with *CUDA out of memory*, lower `point_max` (then `batch`) in section 3, re-run section 3 and this cell.

In [ ]:
run.train('bench', cfgs['bench'], resume=False, print_every=30)
sec = run.last_speed('bench')
print(f'\n{sec:.2f} s/iteration')
print('EPOCH -> iterations -> hours per training run (batch %d):' % K.batch)
for ep in (50, 100, 200, 300, 500):
    its = iterations(N_TRAIN_ROOMS, K.batch, ep)
    print(f'  EPOCH {ep:4d}: {its:6d} iterations = {its * sec / 3600:5.1f} h')
print('Training runs needed: A, B, D (+ H in Part 4) plus two short Phase 0 runs. Set EPOCH in section 3 (same value for all runs).')

## 5. Phase 0 gate (recommended): does sparsity actually hurt PTv3?
Two short runs (same tiny schedule): a dense-trained model tested on dense and on 25 % sparse data, and a sparse-trained model.
**Rule:** if *dense-trained on sparse* is within about 2 mIoU of *dense on dense*, PTv3's own 2 cm voxelisation hides your sparsification: use a coarser sparsity (e.g. 6–10 %) before continuing. Skip this section only if you are sure.

In [ ]:
EP0, EV0 = 20, 5
cfgs['P0_dense'] = experiments.mk('P0_dense', D, D, K, epoch=EP0, eval_epoch=EV0)
cfgs['P0_sparse25'] = experiments.mk('P0_sparse25', S25, S25, K, epoch=EP0, eval_epoch=EV0)
run.train('P0_dense', cfgs['P0_dense'])
run.train('P0_sparse25', cfgs['P0_sparse25'])
r_dd = run.evaluate_run('P0_dense', 'dense', D, 'P0_dense_on_dense')
r_ds = run.evaluate_run('P0_dense', 'sparse25', S25, 'P0_dense_on_sparse25')
r_ss = run.evaluate_run('P0_sparse25', 'sparse25', S25, 'P0_sparse25_on_sparse25')
print(f"\ndense->dense {r_dd['mIoU']*100:.1f} | dense->sparse25 {r_ds['mIoU']*100:.1f} | sparse->sparse25 {r_ss['mIoU']*100:.1f}")
print(f"mIoU drop caused by sparsity: {(r_dd['mIoU'] - r_ds['mIoU'])*100:.1f} points")

## 6. Row A (upper bound) and row C (robustness baseline)
Train PTv3 on dense S3DIS (Areas 1,2,3,4,6). Then test the **same checkpoint** on dense Area 5 (A) and on 25 % sparse Area 5 (C).
`run.train` is resumable: after any interruption re-run this cell and it continues from the last checkpoint.

In [ ]:
run.train('A_dense', cfgs['A_dense'])
rA = run.evaluate_run('A_dense', 'dense', D, 'A_dense_on_dense')
rC = run.evaluate_run('A_dense', 'sparse25', S25, 'C_dense_model_on_sparse25')
print(f"A: {rA['mIoU']*100:.1f} mIoU | C: {rC['mIoU']*100:.1f} mIoU")

## 7. Row B (lower bound): train and test on sparse data

In [ ]:
run.train('B_sparse25', cfgs['B_sparse25'])
rB = run.evaluate_run('B_sparse25', 'sparse25', S25, 'B_sparse25_on_sparse25')
print(f"B: {rB['mIoU']*100:.1f} mIoU")

## 8. Row D (augmentation baseline): dense training with random density augmentation, tested on sparse

In [ ]:
run.train('D_dense_aug', cfgs['D_dense_aug'])
rD = run.evaluate_run('D_dense_aug', 'sparse25', S25, 'D_dense_aug_on_sparse25')
print(f"D: {rD['mIoU']*100:.1f} mIoU")

## 9. Results so far

In [ ]:
rows = results.collect()
print(results.to_markdown(rows))